In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.voos.checkpoints")

ORIGEM = "/Volumes/workspace/voos/raw/"
CHECKPOINT = "/Volumes/workspace/voos/checkpoints/bronze_posicoes"
TABELA = "workspace.voos_bronze.posicoes_raw"

# Estrutura do JSON da OpenSky: cada aviao vem como uma lista de valores mistos,
# por isso na bronze tudo fica como texto (a tipagem e feita na silver)
schema = StructType([
    StructField("time", LongType()),
    StructField("states", ArrayType(ArrayType(StringType()))),
    StructField("_coletado_em", StringType()),
])

df = (
    spark.readStream.format("cloudFiles")          # Auto Loader
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load(ORIGEM)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    df.writeStream
    .option("checkpointLocation", CHECKPOINT)       # guarda quais arquivos ja foram lidos
    .trigger(availableNow=True)                     # processa o que tem de novo e para
    .toTable(TABELA)
    .awaitTermination()
)
print("Bronze atualizada")

Bronze atualizada


In [0]:
display(spark.sql(f"""
    SELECT _source_file, time, size(states) AS qtd_avioes, _ingested_at
    FROM {TABELA}
    ORDER BY time
"""))

_source_file,time,qtd_avioes,_ingested_at
/Volumes/workspace/voos/raw/2026-10-16/posicao_222741.json,1792189657,176,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_222942.json,1792189776,181,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_223144.json,1792189902,178,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_223345.json,1792190022,173,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_223432.json,1792190069,178,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_223634.json,1792190189,182,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_223835.json,1792190314,189,2026-10-16T22:41:23.733Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,1792191504,166,2026-10-16T23:01:38.679Z
/Volumes/workspace/voos/raw/2026-10-16/posicao_230028.json,1792191624,165,2026-10-16T23:01:38.679Z
